# Visualizing Neo4j Graphs

Below is a basic connection to a Neo4j database. We use the `Result.graph` result transformer to map the result to a graph object and store the result in `result`.

In [2]:
%pip install neo4j
%pip install neo4j-viz

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Let us start by setting up a connection to our database.

In [4]:
import os


URI = "neo4j://localhost:7687"

auth = ("neo4j", "password")

To illustrate the library, we will create a small example graph representing web pages.

In [5]:
from neo4j import GraphDatabase

with GraphDatabase.driver(URI, auth=auth) as driver:
    driver.verify_connectivity()
    driver.execute_query(
        """
        CREATE
         (dan:Person {name: 'Dan'}),
         (annie:Person {name: 'Annie'}),
         (matt:Person {name: 'Matt'}),
         (jeff:Person {name: 'Jeff'}),
         (brie:Person {name: 'Brie'}),
         (elsa:Person {name: 'Elsa'}),

         (cookies:Product {name: 'Cookies'}),
         (tomatoes:Product {name: 'Tomatoes'}),
         (cucumber:Product {name: 'Cucumber'}),
         (celery:Product {name: 'Celery'}),
         (kale:Product {name: 'Kale'}),
         (milk:Product {name: 'Milk'}),
         (chocolate:Product {name: 'Chocolate'}),

         (dan)-[:BUYS {amount: 1.2}]->(cookies),
         (dan)-[:BUYS {amount: 3.2}]->(milk),
         (dan)-[:BUYS {amount: 2.2}]->(chocolate),

         (annie)-[:BUYS {amount: 1.2}]->(cucumber),
         (annie)-[:BUYS {amount: 3.2}]->(milk),
         (annie)-[:BUYS {amount: 3.2}]->(tomatoes),

         (matt)-[:BUYS {amount: 3}]->(tomatoes),
         (matt)-[:BUYS {amount: 2}]->(kale),
         (matt)-[:BUYS {amount: 1}]->(cucumber),

         (jeff)-[:BUYS {amount: 3}]->(cookies),
         (jeff)-[:BUYS {amount: 2}]->(milk),

         (brie)-[:BUYS {amount: 1}]->(tomatoes),
         (brie)-[:BUYS {amount: 2}]->(milk),
         (brie)-[:BUYS {amount: 2}]->(kale),
         (brie)-[:BUYS {amount: 3}]->(cucumber),
         (brie)-[:BUYS {amount: 0.3}]->(celery),

         (elsa)-[:BUYS {amount: 3}]->(chocolate),
         (elsa)-[:BUYS {amount: 3}]->(milk)
    """
    )

ConstraintError: {code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Node(171) already exists with label `Person` and property `name` = 'Dan'}

We can now fetch data from our database, to later include in our visualization.

In [6]:
from neo4j import Result, RoutingControl

with GraphDatabase.driver(URI, auth=auth) as driver:
    driver.verify_connectivity()

    result = driver.execute_query(
        "MATCH (n)-[r]->(m) RETURN n,r,m",
        database_="neo4j",
        routing_=RoutingControl.READ,
        result_transformer_=Result.graph,
    )

    result

print(
    f"Result graph has: {len(result.nodes)} nodes, {len(result.relationships)} relationships"
)

Result graph has: 184 nodes, 271 relationships


Below we map the graph object's nodes and relationships to the correct format for NVL. The Node and Relationship type documentation can be found in NVL's docs: https://neo4j.com/docs/nvl/current/base-library/#_nodes

Now, we can render our result with NVL the following way:


In [9]:
from neo4j_viz.neo4j import from_neo4j

VG = from_neo4j(result)

VG.render()

In [11]:
VG.color_nodes(field="caption")

In [12]:
VG.render()

Lastly we clean up our database by removing our toy graph.

In [ ]:
with GraphDatabase.driver(URI, auth=auth) as driver:
    result = driver.execute_query(
        "MATCH (n:Person|Product) DETACH DELETE n RETURN count(n) "
    )
    print(result.summary.counters)

**NOTE:** Since in this example we didn't already have a Neo4j DB populated with data, it would actually have been more convenient to use the serverless `from_gql_create` importer method to create our `VisualizationGraph`.